# Module 8 Guided Lab: Data Integration with Pandas Merge

**BAN 6003: Data Management and Analytics Integration**

This week we focus on data integration using Pandas. In previous modules, we worked mostly within one table. Now we start combining related datasets.

The key idea for this module is:

> Data integration is not just putting tables together. It is making sure the combined result still represents what you think it represents.

We will use the `nycflights13` data because it includes multiple related tables: `flights`, `airlines`, `airports`, `planes`, and `weather`.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Explain keys, granularity, and relationships between datasets.
2. Identify primary key and foreign key candidates.
3. Use `pd.merge()` to combine tables.
4. Use `how="left"`, `how="inner"`, `how="outer"`, and `how="right"` appropriately.
5. Use `pd.concat()` to stack similar datasets.
6. Check row counts before and after merges.
7. Detect duplicate keys before merging.
8. Explain merge results in business language.

We will keep the code practical. We will not use complex loops or custom functions.

## Business Scenario

Imagine you are helping an airport analytics team prepare flight data for reporting.

The `flights` table tells us what happened for each flight. But it uses coded values such as carrier codes and airport codes.

For example, `carrier = "UA"` is not easy for a business audience unless we attach the airline name. `dest = "LAX"` is more useful if we also attach airport details. `tailnum` can connect a flight to aircraft details in the planes table.

This is why integration matters. One table rarely contains everything we need.

## 0. Setup: Load Packages and Tables

If `nycflights13` is not installed in your environment, the following cell will install it.

### Package setup note

If you are using **GitHub Codespaces**, the required packages should already be installed from `requirements.txt` when the Codespace is created. You usually do not need to run any `%pip install` command.

If you are running the repository on your **local computer** and an import fails, uncomment and run the `%pip install` line(s) in the setup cell below, then rerun the imports.



In [ ]:
# Codespaces: nycflights13 is installed from requirements.txt.
# Local only: if the import below fails, uncomment and run this line:
# %pip install nycflights13

import nycflights13
import pandas as pd


In [ ]:
flights = nycflights13.flights.copy()
airlines = nycflights13.airlines.copy()
airports = nycflights13.airports.copy()
planes = nycflights13.planes.copy()
weather = nycflights13.weather.copy()

Let's take a quick look at the tables we will use.

In [ ]:
flights.head()

In [ ]:
airlines.head()

In [ ]:
airports.head()

In [ ]:
planes.head()

## 1. Keys and Relationships

Before merging tables, we need to ask two questions:

1. What column connects the tables?
2. Is the key unique in the table we are merging from?

A **primary key** uniquely identifies rows in a table. A **foreign key** is a column in another table that refers to that primary key.

Example: `airlines["carrier"]` should identify one airline. `flights["carrier"]` refers to that airline code many times.

### Check key uniqueness in the lookup table

Before merging `flights` with `airlines`, check whether `carrier` is unique in the `airlines` table.

In [ ]:
airlines["carrier"].duplicated().sum()

In [ ]:
airlines["carrier"].nunique(), airlines.shape[0]

If the number of unique carrier codes equals the number of rows, then `carrier` is unique in the `airlines` table.

Now check the `flights` table. We expect carrier codes to repeat there because there are many flights for each carrier.

In [ ]:
flights["carrier"].duplicated().sum()

In [ ]:
flights["carrier"].value_counts().head()

This is normal. The same airline appears on many flights. That means the relationship is many flights to one airline.

### Your Turn 1

Check whether `faa` is unique in the `airports` table. Then check how often `dest` appears in the `flights` table.

In [ ]:
# Your Turn 1A
# Check whether airports["faa"] is unique.

In [ ]:
# Your Turn 1B
# Check how often destination airport codes appear in flights["dest"].

Write one sentence explaining the relationship between `flights["dest"]` and `airports["faa"]`.

**Your answer:**  
Type your sentence here.

## 2. Basic `pd.merge()` Pattern

The basic merge pattern is:

```python
pd.merge(left_table, right_table, how="left", left_on="column_in_left", right_on="column_in_right")
```

Important arguments:

- `left_table`: the main table you want to keep.
- `right_table`: the table you want to add information from.
- `how`: the type of join.
- `left_on`: the key column in the left table.
- `right_on`: the key column in the right table.

If the key column has the same name in both tables, use `on="key_column"`.

## 3. Left Join: Keep All Rows from the Main Table

### Business question

Can we attach airline names to every flight record?

We want to keep all flights and add the full airline name from the `airlines` table. This is a classic left join.

In [ ]:
flights_with_airline = pd.merge(
    flights,
    airlines,
    how="left",
    on="carrier"
)

flights_with_airline[["carrier", "name", "flight", "origin", "dest"]].head()

### Check row counts after a left join

A left join should keep all rows from the left table. Here, the left table is `flights`. The row count should remain the same unless the right table has duplicate keys that create extra matches.

In [ ]:
flights.shape[0], flights_with_airline.shape[0]

In [ ]:
flights_with_airline["name"].isna().sum()

The row count check tells us whether the merge unexpectedly created or lost rows. The missing value check tells us whether any flights had carrier codes that did not match the airline lookup table.

### Your Turn 2

Use a left join to attach destination airport information to the flights table.

Use:

- left table: `flights`
- right table: `airports`
- left key: `dest`
- right key: `faa`

Name the result `flights_with_dest_airport`.

In [ ]:
# Your Turn 2
# Left join flights with airports using dest and faa.

In [ ]:
# Your Turn 2 check
# Compare row counts before and after the merge.

In [ ]:
# Your Turn 2 check
# Check how many rows have missing airport name information after the merge.

## 4. Inner Join: Keep Only Matching Rows

### Business question

What if we only want flight records that have matching airline information?

An inner join keeps only rows where the key appears in both tables.

In [ ]:
flights_inner_airline = pd.merge(
    flights,
    airlines,
    how="inner",
    on="carrier"
)

flights.shape[0], flights_inner_airline.shape[0]

In this case, the row count may be the same because all carriers in `flights` likely exist in `airlines`. But do not assume this will always happen. Inner joins can remove records. That can be correct, but it should be intentional.

### Example: inner join with planes

The `planes` table contains aircraft details by `tailnum`. Some flights may not have a matching `tailnum` in the `planes` table. Let's compare a left join and an inner join.

In [ ]:
flights_planes_left = pd.merge(
    flights,
    planes,
    how="left",
    on="tailnum",
    suffixes=("_flight", "_plane")
)

flights_planes_inner = pd.merge(
    flights,
    planes,
    how="inner",
    on="tailnum",
    suffixes=("_flight", "_plane")
)

flights.shape[0], flights_planes_left.shape[0], flights_planes_inner.shape[0]

In [ ]:
flights_planes_left[["tailnum", "carrier", "flight", "manufacturer", "model", "seats"]].head()

The inner join can reduce the dataset because it keeps only flights that have a matching aircraft record. In business work, this matters. If missing aircraft information is not random, an inner join may remove important records.

### Your Turn 3

Use an inner join to attach destination airport information to flights. Compare the row count with the original flights table. Did the inner join remove any rows?

In [ ]:
# Your Turn 3
# Inner join flights with airports using dest and faa.

In [ ]:
# Your Turn 3 check
# Compare row counts.

**Your answer:**  
Did the inner join remove any rows? What does that suggest?

## 5. Outer Join: Keep Everything from Both Tables

### Business question

What if we want to audit whether there are airport codes that appear in one table but not the other?

An outer join keeps all rows from both tables. It is often useful for checking mismatches.

In [ ]:
flight_dest_codes = flights[["dest"]].drop_duplicates().rename(columns={"dest": "faa"})

airport_audit = pd.merge(
    flight_dest_codes,
    airports[["faa", "name", "lat", "lon"]],
    how="outer",
    on="faa",
    indicator=True
)

airport_audit["_merge"].value_counts()

In [ ]:
airport_audit.head()

The `indicator=True` argument adds a special `_merge` column. This column tells us whether each row came from both tables, the left table only, or the right table only. This is useful for audit-style checks.

In [ ]:
airport_audit.query("_merge != 'both'").head(10)

### Your Turn 4

Create a list of unique origin airport codes from `flights`, rename the column to `faa`, and outer join it with `airports`. Use `indicator=True`. Then count the `_merge` categories.

In [ ]:
# Your Turn 4
# Outer join unique origin airport codes with airports.

## 6. Right Join: Keep All Rows from the Right Table

Right joins are less common in Pandas workflows because we can usually switch the table order and use a left join. Still, it is helpful to understand what it does. A right join keeps all rows from the right table and matches rows from the left table when possible.

In [ ]:
right_join_example = pd.merge(
    flights[["carrier", "flight", "origin", "dest"]].head(100),
    airlines,
    how="right",
    on="carrier"
)

right_join_example.head()

In [ ]:
right_join_example.shape

In practice, I usually recommend using left joins when possible because they make the main table very clear. If your main table is `flights`, put `flights` on the left and use `how="left"`.

## 7. Many-to-Many Risk: Duplicate Keys Can Expand Rows

One common merge mistake is merging with a table that has duplicate keys. If the right table has multiple rows for the same key, one left row may match multiple right rows. This can expand the row count.

### A small example

Let's create a small fake lookup table with duplicate carrier keys.

In [ ]:
fake_airline_regions = pd.DataFrame({
    "carrier": ["AA", "AA", "DL", "UA"],
    "region_label": ["legacy", "large", "legacy", "legacy"]
})

fake_airline_regions

In [ ]:
small_flights = flights[["carrier", "flight", "origin", "dest"]].query("carrier in ['AA', 'DL', 'UA']").head(10)

small_flights

In [ ]:
merged_fake = pd.merge(
    small_flights,
    fake_airline_regions,
    how="left",
    on="carrier"
)

small_flights.shape[0], merged_fake.shape[0]

In [ ]:
merged_fake.head(10)

The row count increased because `AA` matched two rows in the fake lookup table. This is why we check duplicate keys before merging.

In [ ]:
fake_airline_regions["carrier"].duplicated().sum()

### Your Turn 5

Check whether `tailnum` is unique in the `planes` table. Then explain whether merging `flights` with `planes` on `tailnum` is likely to create unexpected row expansion.

In [ ]:
# Your Turn 5A
# Check whether tailnum is unique in planes.

In [ ]:
# Your Turn 5B
# Optional: compare row counts after a left join with planes.

**Your answer:**  
Type your explanation here.

## 8. `pd.concat()`: Stacking Similar Tables

`pd.merge()` combines tables side by side using keys.

`pd.concat()` stacks tables together when they have similar columns.

### Business question

Suppose we want one table containing flights from January and February. Each monthly table has the same columns, so we can stack them.

In [ ]:
jan_flights = flights.query("month == 1")
feb_flights = flights.query("month == 2")

jan_flights.shape, feb_flights.shape

In [ ]:
jan_feb_flights = pd.concat([jan_flights, feb_flights], axis=0)

jan_feb_flights.shape

In [ ]:
jan_feb_flights["month"].value_counts().sort_index()

Important arguments:

- `axis=0` stacks rows. This is the default.
- `axis=1` stacks columns side by side, but it should be used carefully because it aligns by index.
- `ignore_index=True` can reset the row index after stacking.

In [ ]:
jan_feb_flights_clean_index = pd.concat(
    [jan_flights, feb_flights],
    axis=0,
    ignore_index=True
)

jan_feb_flights_clean_index.head()

### Your Turn 6

Create a new DataFrame that stacks March and April flights. Then check the row count and month counts.

In [ ]:
# Your Turn 6
# Stack March and April flights using pd.concat().

In [ ]:
# Your Turn 6 check
# Check row count and month counts.

## 9. Integration Checklist

Before and after integration, use this checklist:

1. What is the main table?
2. What is the lookup or secondary table?
3. What key connects the tables?
4. Is the key unique where it should be unique?
5. What type of join is appropriate?
6. Did the row count change as expected?
7. Did the merge create missing values?
8. Can you explain the result in business language?

## 10. Final Practice: Build an Integrated Flight Table

### Business question

Create a flight-level dataset that is easier for business users to understand.

Start with a smaller set of columns from `flights`, then integrate:

- airline names from `airlines`
- destination airport names from `airports`

Use left joins so that the main table remains flight-level.

In [ ]:
# Final Practice Step 1
# Create a smaller flight table with selected columns.

flight_base = flights[[
    "year", "month", "day", "carrier", "flight", "origin", "dest",
    "dep_delay", "arr_delay", "distance", "air_time"
]].copy()

flight_base.head()

In [ ]:
# Final Practice Step 2
# Merge airline names.

In [ ]:
# Final Practice Step 3
# Merge destination airport names.

In [ ]:
# Final Practice Step 4
# Check row counts before and after each merge.

In [ ]:
# Final Practice Step 5
# Check missing values in the added columns.

### Final Reflection

Write 4–6 sentences answering the following:

1. What was the main table in your final practice?
2. What lookup tables did you use?
3. What keys connected the tables?
4. Why did you use left joins?
5. Did the row count change in an expected way?
6. What business value did the integrated table add?

**Your reflection:**  
Type your answer here.

## 11. Save Your Work

Before submitting:

1. Save the notebook.
2. Restart the kernel and run all cells if possible.
3. Make sure all “Your Turn” sections and the final reflection are complete.
4. Commit and push your work to GitHub.

Suggested commands:

```bash
git add .
git commit -m "Complete Module 8 pandas integration lab"
git push
```